# SAM Food Segmentation - Inference Notebook

This notebook provides inference capabilities for the trained SAM model with LoRA adapters.

## Features:
- Load trained checkpoint
- Run inference on single images or batches
- Interactive inference with point prompts
- Batch inference on validation set
- Visualization of predictions
- Metrics computation


## 1. Setup and Imports


In [ ]:
import sys
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image
import json
from typing import Dict, List, Tuple, Optional

# Add project root to path
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from configs.config import Config
from src.models.sam_lora import SAMLoRAModel, SamPredictorLoRA
from src.data.foodseg_dataset import create_data_loaders
from src.utils.metrics import calculate_miou, calculate_dice, calculate_precision_recall_f1
from src.utils.visualization import Visualizer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")


## 2. Configuration


In [ ]:
# ===== CONFIGURATION =====
# Update these paths according to your setup

# Path to your trained checkpoint
CHECKPOINT_PATH = "checkpoints/best_model.pth"  # Update this!

# Path to SAM base model checkpoint (required for loading)
SAM_CHECKPOINT_PATH = None  # Set this if not in config

# Model architecture (must match training)
MODEL_NAME = "vit_b"  # Options: vit_b, vit_l, vit_h

# Dataset path (for batch inference)
DATASET_PATH = None  # Will use default if None

# Device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Output directory for inference results
OUTPUT_DIR = "inference_results"

print(f"Checkpoint path: {CHECKPOINT_PATH}")
print(f"Device: {DEVICE}")
print(f"Model: {MODEL_NAME}")


## 2.5. Download Model Weights (if needed)


In [ ]:
# Download model weights from Hugging Face and load into memory
import urllib.request
import io

MODEL_URL = "https://huggingface.co/JasonGong111/SAM4Food/resolve/main/best_model.pth"

print(f"Downloading model weights from {MODEL_URL}...")
print("This may take a few minutes depending on your internet connection...")

try:
    # Download into memory buffer with progress tracking
    req = urllib.request.urlopen(MODEL_URL)
    total_size = int(req.headers.get('Content-Length', 0))
    
    downloaded = 0
    chunk_size = 8192
    model_data = io.BytesIO()
    
    while True:
        chunk = req.read(chunk_size)
        if not chunk:
            break
        model_data.write(chunk)
        downloaded += len(chunk)
        
        if total_size > 0:
            percent = (downloaded / total_size) * 100
            bar_length = 40
            filled_length = int(bar_length * downloaded // total_size)
            bar = '=' * filled_length + '-' * (bar_length - filled_length)
            print(f'\r[{bar}] {percent:.1f}% ({downloaded}/{total_size} bytes)', end='', flush=True)
    
    print(f"\n✓ Model weights downloaded successfully")
    print(f"  File size: {downloaded / (1024 * 1024):.2f} MB")
    
    # Load checkpoint from memory
    model_data.seek(0)  # Reset buffer position
    checkpoint = torch.load(model_data, map_location=DEVICE)
    
except Exception as e:
    print(f"\n✗ Error downloading model weights: {e}")
    print(f"Please check your internet connection and try again.")
    raise


## 3. Load Configuration and Model


In [ ]:
# Initialize configuration
import urllib.request
import tempfile

config = Config()
config.model.sam_model_name = MODEL_NAME
config.system.device = DEVICE

# Set dataset path if provided
if DATASET_PATH:
    config.data.dataset_path = DATASET_PATH

# Create output directory
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

# Download base SAM checkpoint if not provided (required for model initialization)
# Note: Even though your checkpoint contains all weights, the model architecture 
# needs the base SAM model to be loaded first to set up the structure
if SAM_CHECKPOINT_PATH:
    config.model.sam_checkpoint_path = SAM_CHECKPOINT_PATH
elif config.model.sam_checkpoint_path is None:
    # Auto-download base SAM checkpoint based on model name
    sam_checkpoints = {
        'vit_b': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth',
        'vit_l': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_l_0b3195.pth',
        'vit_h': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth'
    }
    
    sam_url = sam_checkpoints.get(MODEL_NAME)
    if sam_url:
        # Create temporary file for base SAM checkpoint
        temp_dir = Path(tempfile.gettempdir()) / "sam_checkpoints"
        temp_dir.mkdir(parents=True, exist_ok=True)
        sam_checkpoint_path = temp_dir / f"sam_{MODEL_NAME}.pth"
        
        if not sam_checkpoint_path.exists():
            print(f"Downloading base SAM {MODEL_NAME} checkpoint...")
            print(f"This is required for model initialization (LoRA adapters modify the base model)")
            
            try:
                req = urllib.request.urlopen(sam_url)
                total_size = int(req.headers.get('Content-Length', 0))
                
                downloaded = 0
                chunk_size = 8192
                
                with open(sam_checkpoint_path, 'wb') as f:
                    while True:
                        chunk = req.read(chunk_size)
                        if not chunk:
                            break
                        f.write(chunk)
                        downloaded += len(chunk)
                        
                        if total_size > 0:
                            percent = (downloaded / total_size) * 100
                            bar_length = 40
                            filled_length = int(bar_length * downloaded // total_size)
                            bar = '=' * filled_length + '-' * (bar_length - filled_length)
                            print(f'\r[{bar}] {percent:.1f}% ({downloaded}/{total_size} bytes)', end='', flush=True)
                
                print(f"\n✓ Base SAM checkpoint downloaded to temporary location")
            except Exception as e:
                print(f"\n✗ Error downloading base SAM checkpoint: {e}")
                raise
        
        config.model.sam_checkpoint_path = str(sam_checkpoint_path)
        print(f"Using base SAM checkpoint: {sam_checkpoint_path}")
    else:
        raise ValueError(f"Unknown model name: {MODEL_NAME}")

print("Configuration loaded successfully!")


In [ ]:
# Load model
print("Loading model...")
model = SAMLoRAModel(config).to(DEVICE)

# Load trained checkpoint from memory (downloaded in previous cell)
print("Loading checkpoint from memory...")

# Handle different checkpoint formats
if 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded checkpoint from epoch {checkpoint.get('epoch', 'unknown')}")
else:
    model.load_state_dict(checkpoint)
    print("Loaded checkpoint (state dict only)")

# Print checkpoint info if available
if 'lora_config' in checkpoint:
    print(f"LoRA config: {checkpoint['lora_config']}")

model.eval()
print("Model loaded and set to evaluation mode!")


## 4. Helper Functions for Inference


In [ ]:
def preprocess_image(image_path: str, target_size: int = 1024) -> Tuple[torch.Tensor, Tuple[int, int]]:
    """
    Preprocess image for SAM inference
    
    Args:
        image_path: Path to image file
        target_size: Target size for resizing
    
    Returns:
        Preprocessed image tensor and original size
    """
    # Load image
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError(f"Could not load image from {image_path}")
    
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    original_size = image_rgb.shape[:2]  # (H, W)
    
    # Resize image
    h, w = original_size
    scale = min(target_size / h, target_size / w)
    new_h, new_w = int(h * scale), int(w * scale)
    
    image_resized = cv2.resize(image_rgb, (new_w, new_h))
    
    # Pad to square
    padded_image = np.zeros((target_size, target_size, 3), dtype=np.uint8)
    start_h = (target_size - new_h) // 2
    start_w = (target_size - new_w) // 2
    padded_image[start_h:start_h + new_h, start_w:start_w + new_w] = image_resized
    
    # Normalize
    mean = np.array(config.data.mean).reshape(1, 1, 3)
    std = np.array(config.data.std).reshape(1, 1, 3)
    
    image_normalized = (padded_image.astype(np.float32) / 255.0 - mean) / std
    
    # Convert to tensor
    image_tensor = torch.from_numpy(image_normalized).permute(2, 0, 1).unsqueeze(0).float()
    
    return image_tensor, original_size


def predict_with_points(
    model: SAMLoRAModel,
    image_tensor: torch.Tensor,
    point_coords: np.ndarray,
    point_labels: np.ndarray,
    original_size: Tuple[int, int],
    device: str = "cuda"
) -> np.ndarray:
    """
    Predict mask given point prompts
    
    Args:
        model: Trained SAM model
        image_tensor: Preprocessed image tensor [1, 3, H, W]
        point_coords: Point coordinates [[x1, y1], [x2, y2], ...]
        point_labels: Point labels [1, 0, ...] (1=foreground, 0=background)
        original_size: Original image size (H, W)
        device: Device to use
    
    Returns:
        Predicted mask as numpy array [H, W]
    """
    model.eval()
    
    with torch.no_grad():
        # Move image to device
        image_tensor = image_tensor.to(device)
        
        # Get image embeddings
        image_features = model.image_encoder(image_tensor)
        
        # Prepare prompts
        # Convert point coordinates to tensor format
        # SAM expects coordinates in the resized image space
        point_coords_tensor = torch.from_numpy(point_coords).float().unsqueeze(0).to(device)
        point_labels_tensor = torch.from_numpy(point_labels).int().unsqueeze(0).to(device)
        
        prompts = {
            'points': point_coords_tensor,
            'point_labels': point_labels_tensor
        }
        
        # Get prediction
        pred_masks = model.sam_model(image_features, prompts)
        
        # Resize to original size
        pred_masks = model.resize_predictions(pred_masks, original_size)
        
        # Convert to numpy
        pred_mask = torch.sigmoid(pred_masks[0, 0]).cpu().numpy()
        
    return pred_mask


def visualize_prediction(
    image: np.ndarray,
    mask: np.ndarray,
    pred_mask: np.ndarray,
    point_coords: Optional[np.ndarray] = None,
    point_labels: Optional[np.ndarray] = None,
    threshold: float = 0.5
) -> plt.Figure:
    """
    Visualize prediction with overlay
    
    Args:
        image: Original image [H, W, 3]
        mask: Ground truth mask [H, W] (optional)
        pred_mask: Predicted mask [H, W]
        point_coords: Point coordinates for visualization
        point_labels: Point labels
        threshold: Threshold for binary mask
    
    Returns:
        Matplotlib figure
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Original image
    axes[0].imshow(image)
    axes[0].set_title('Original Image', fontweight='bold')
    axes[0].axis('off')
    
    # Add point prompts if provided
    if point_coords is not None:
        for coord, label in zip(point_coords, point_labels):
            color = 'green' if label == 1 else 'red'
            marker = '+' if label == 1 else 'x'
            axes[0].scatter(coord[0], coord[1], c=color, marker=marker, s=100, linewidths=2)
    
    # Ground truth mask (if available)
    if mask is not None:
        axes[1].imshow(mask, cmap='gray')
        axes[1].set_title('Ground Truth Mask', fontweight='bold')
    else:
        axes[1].imshow(image)
        axes[1].set_title('Original Image', fontweight='bold')
    axes[1].axis('off')
    
    # Prediction overlay
    pred_binary = (pred_mask > threshold).astype(np.float32)
    overlay = image.copy().astype(np.float32) / 255.0
    
    # Add mask overlay
    mask_color = np.array([0, 1, 0])  # Green
    overlay = overlay * (1 - 0.5 * pred_binary[..., None]) + mask_color * (0.5 * pred_binary[..., None])
    
    axes[2].imshow(np.clip(overlay, 0, 1))
    axes[2].set_title(f'Prediction (Threshold={threshold})', fontweight='bold')
    axes[2].axis('off')
    
    plt.tight_layout()
    return fig


print("Helper functions defined!")


## 5. Single Image Inference (Interactive)


In [ ]:
# ===== SINGLE IMAGE INFERENCE =====
# Update this path to your test image
IMAGE_PATH = "path/to/your/image.jpg"  # Update this!

# Point prompts: [[x1, y1], [x2, y2], ...]
# Labels: 1 for foreground (food), 0 for background
# Example: Click on the food item in the image
POINT_COORDS = np.array([[500, 400]])  # Update with your point coordinates!
POINT_LABELS = np.array([1])  # 1 = foreground point

if Path(IMAGE_PATH).exists():
    # Preprocess image
    print(f"Loading image from {IMAGE_PATH}...")
    image_tensor, original_size = preprocess_image(IMAGE_PATH, target_size=config.data.input_size)
    
    # Load original image for visualization
    original_image = cv2.imread(IMAGE_PATH)
    original_image = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)
    
    # Run prediction
    print("Running prediction...")
    pred_mask = predict_with_points(
        model, image_tensor, POINT_COORDS, POINT_LABELS, original_size, DEVICE
    )
    
    # Visualize
    fig = visualize_prediction(
        original_image, None, pred_mask, POINT_COORDS, POINT_LABELS
    )
    plt.show()
    
    # Save result
    save_path = output_dir / f"prediction_{Path(IMAGE_PATH).stem}.png"
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"Result saved to {save_path}")
else:
    print(f"Image not found at {IMAGE_PATH}")
    print("Please update IMAGE_PATH and POINT_COORDS above.")


## 6. Batch Inference on Validation Set


In [ ]:
# ===== BATCH INFERENCE =====
# Run inference on validation set

NUM_SAMPLES = 20  # Number of samples to process
SAVE_VISUALIZATIONS = True

# Create data loader
try:
    _, val_loader = create_data_loaders(config)
    print(f"Validation loader created with {len(val_loader.dataset)} samples")
    
    # Initialize metrics
    all_miou = []
    all_dice = []
    all_precision = []
    all_recall = []
    all_f1 = []
    
    model.eval()
    
    with torch.no_grad():
        for batch_idx, batch in enumerate(val_loader):
            if batch_idx * config.training.batch_size >= NUM_SAMPLES:
                break
            
            # Move batch to device
            image = batch['image'].to(DEVICE)
            mask = batch['mask'].to(DEVICE)
            
            # Get predictions
            image_features = model.image_encoder(image)
            prompts = {
                'points': batch['prompts']['points'].to(DEVICE),
                'point_labels': batch['prompts']['point_labels'].to(DEVICE)
            }
            pred_masks = model.sam_model(image_features, prompts)
            pred_masks = model.resize_predictions(pred_masks, mask.shape[-2:])
            pred_masks_sigmoid = torch.sigmoid(pred_masks.squeeze(1))
            
            # Calculate metrics for each sample in batch
            batch_size = image.size(0)
            for i in range(batch_size):
                sample_idx = batch_idx * config.training.batch_size + i
                if sample_idx >= NUM_SAMPLES:
                    break
                
                pred_i = pred_masks_sigmoid[i:i+1]
                mask_i = mask[i:i+1]
                
                # Calculate metrics
                miou = calculate_miou(pred_i, mask_i).item()
                dice = calculate_dice(pred_i, mask_i).item()
                prf = calculate_precision_recall_f1(pred_i, mask_i)
                
                all_miou.append(miou)
                all_dice.append(dice)
                all_precision.append(prf['precision'])
                all_recall.append(prf['recall'])
                all_f1.append(prf['f1'])
                
                # Save visualization if requested
                if SAVE_VISUALIZATIONS:
                    # Denormalize image
                    img_i = image[i].cpu()
                    mean = torch.tensor(config.data.mean).view(3, 1, 1)
                    std = torch.tensor(config.data.std).view(3, 1, 1)
                    img_denorm = img_i * std + mean
                    img_denorm = torch.clamp(img_denorm, 0, 1)
                    img_np = img_denorm.permute(1, 2, 0).numpy()
                    
                    mask_np = mask_i[0].cpu().numpy()
                    pred_np = pred_i[0].cpu().numpy()
                    
                    fig = visualize_prediction(img_np, mask_np, pred_np)
                    save_path = output_dir / f"batch_prediction_{sample_idx:03d}.png"
                    fig.savefig(save_path, dpi=150, bbox_inches='tight')
                    plt.close(fig)
                
                if (sample_idx + 1) % 10 == 0:
                    print(f"Processed {sample_idx + 1}/{NUM_SAMPLES} samples")
    
    # Print summary statistics
    print("\n" + "="*50)
    print("INFERENCE RESULTS")
    print("="*50)
    print(f"Number of samples: {len(all_miou)}")
    print(f"Mean IoU: {np.mean(all_miou):.4f} ± {np.std(all_miou):.4f}")
    print(f"Mean Dice: {np.mean(all_dice):.4f} ± {np.std(all_dice):.4f}")
    print(f"Mean Precision: {np.mean(all_precision):.4f} ± {np.std(all_precision):.4f}")
    print(f"Mean Recall: {np.mean(all_recall):.4f} ± {np.std(all_recall):.4f}")
    print(f"Mean F1: {np.mean(all_f1):.4f} ± {np.std(all_f1):.4f}")
    
    # Save metrics to JSON
    metrics_dict = {
        'num_samples': len(all_miou),
        'mean_iou': float(np.mean(all_miou)),
        'std_iou': float(np.std(all_miou)),
        'mean_dice': float(np.mean(all_dice)),
        'std_dice': float(np.std(all_dice)),
        'mean_precision': float(np.mean(all_precision)),
        'mean_recall': float(np.mean(all_recall)),
        'mean_f1': float(np.mean(all_f1)),
        'per_sample_metrics': {
            'miou': [float(x) for x in all_miou],
            'dice': [float(x) for x in all_dice],
            'precision': [float(x) for x in all_precision],
            'recall': [float(x) for x in all_recall],
            'f1': [float(x) for x in all_f1]
        }
    }
    
    metrics_path = output_dir / "inference_metrics.json"
    with open(metrics_path, 'w') as f:
        json.dump(metrics_dict, f, indent=2)
    print(f"\nMetrics saved to {metrics_path}")
    
except Exception as e:
    print(f"Error during batch inference: {e}")
    print("Make sure the dataset path is correctly configured.")


## 7. Visualization of Metrics Distribution


In [ ]:
# Plot metrics distribution
if len(all_miou) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle('Inference Metrics Distribution', fontsize=16, fontweight='bold')
    
    metrics_data = [
        (all_miou, 'mIoU', 'Mean IoU'),
        (all_dice, 'Dice', 'Dice Coefficient'),
        (all_precision, 'Precision', 'Precision'),
        (all_recall, 'Recall', 'Recall'),
        (all_f1, 'F1', 'F1 Score')
    ]
    
    for idx, (data, name, title) in enumerate(metrics_data):
        row = idx // 3
        col = idx % 3
        
        axes[row, col].hist(data, bins=20, alpha=0.7, edgecolor='black')
        axes[row, col].axvline(np.mean(data), color='red', linestyle='--', 
                              label=f'Mean: {np.mean(data):.3f}')
        axes[row, col].set_title(title)
        axes[row, col].set_xlabel(name)
        axes[row, col].set_ylabel('Frequency')
        axes[row, col].legend()
        axes[row, col].grid(True, alpha=0.3)
    
    # Remove empty subplot
    axes[1, 2].axis('off')
    
    plt.tight_layout()
    
    save_path = output_dir / "metrics_distribution.png"
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Metrics distribution plot saved to {save_path}")
else:
    print("No metrics data available. Run batch inference first.")


## 8. Using the Visualizer Class (Alternative Method)


In [ ]:
# Use the built-in visualizer for comprehensive visualization
try:
    _, val_loader = create_data_loaders(config)
    
    visualizer = Visualizer(config, save_dir=str(output_dir / "visualizations"))
    
    viz_dir = visualizer.visualize_predictions(
        model, val_loader, DEVICE, num_samples=10, save_prefix="inference"
    )
    
    print(f"\nVisualizations saved to: {viz_dir}")
except Exception as e:
    print(f"Error using visualizer: {e}")
    print("Make sure the dataset path is correctly configured.")


## 9. Full Evaluation (Using Built-in Functions)


In [ ]:
# Run full evaluation using built-in evaluation function
from src.utils.metrics import evaluate_model

try:
    _, val_loader = create_data_loaders(config)
    
    print("Running full evaluation...")
    results = evaluate_model(model, val_loader, config, DEVICE)
    
    print("\n" + "="*50)
    print("FULL EVALUATION RESULTS")
    print("="*50)
    for metric, value in results.items():
        print(f"{metric}: {value:.4f}")
    
    # Save results
    results_path = output_dir / "full_evaluation_results.json"
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to {results_path}")
    
except Exception as e:
    print(f"Error during evaluation: {e}")
    print("Make sure the dataset path is correctly configured.")
